# 26 · Little's law: from QPS to concurrency
Level L5 · Part 4 · core · ~20 min · builds on 08, 13

Time: about 20 min. No server, no model and no Docker: every number comes from the course data or from a stated assumption, so the notebook runs on a laptop.

## Goal
By the end of this lesson you can turn a peak request rate and a time per request into the number of requests in flight, turn that into replicas with headroom, check the rule with a queue simulation, and show why waits climb steeply when a replica runs close to full.

## Why it matters
- **Without it:** you size a fleet by guessing. The simplest guess is one request per replica at a time. At the desk's peak that means one replica per request in flight, which is far too many machines. The other guess is "it ran fine on one GPU", and that fails at the p95 once arrivals bunch up.
- **With it:** one multiplication gives the requests in flight. One division by the slots a replica really has, at a safe utilisation, gives the replica count. The same rule shows where the queue starts to grow.
- **When you don't need it:** a fixed batch of files that must finish by Friday, where the total work is known and the formula is just a division. Or an overloaded system: the rule describes a steady state, and an overloaded queue never reaches one.

The cell below puts a number on the first point.

In [1]:
PEAK_QPS = 10_000              # the desk's peak requests per second (the course's scale; your number)
SECONDS_PER_REQUEST = 2.0      # ASSUMPTION: a rounded guess at the time one request holds a slot at peak (step 3 measures it: 1.98 s)
naive = PEAK_QPS * SECONDS_PER_REQUEST
print(f"one request per replica at a time, with the 2.0 s guess (rounded): {naive:,.0f} replicas")
print("with batching and Little's law (step 3): about 300 replicas, see the output there")

one request per replica at a time, with the 2.0 s guess (rounded): 20,000 replicas
with batching and Little's law (step 3): about 300 replicas, see the output there


## The idea
**Little's law** says: the average number of requests in a system equals the arrival rate times the average time each one spends there. In symbols, **L = λ × W**. It needs only two things: the system is in a steady state (the queue is not growing without bound), and the averages are taken over the same window. It holds for any arrival pattern and any order of service.

Think of a shop. If 6 customers arrive each minute and each stays 5 minutes, there are on average 30 people inside. You do not need to know who came first or how bursty they were.

For a serving fleet, λ is the peak QPS, W is the seconds a request holds a slot (its prefill plus its answer tokens at the decode speed), and L is the number of requests in flight. L is what the replicas must hold at once.

## Picture
```mermaid
flowchart LR
  A["Peak QPS (λ)"] -->|"× W, seconds per request"| B["Requests in flight (L)"]
  B -->|"÷ slots per replica × target utilisation"| C["Replicas, plus one spare"]
```

## Already know this?
This is the first lesson of Part 4 (capacity and economics). If you can answer these five without looking, skip to lesson 29; use 27 and 28 for load tests and cost.

1. A service gets 6 requests a second and each takes 4 seconds. How many are in flight on average?
2. Why does the lab server of lesson 08 run at most 4 requests at once, and what happens to the rest?
3. Why can a full KV cache limit concurrency below the `max_num_seqs` setting?
4. Why does a load test need Poisson arrivals (random gaps between requests around a steady average rate) rather than a fixed rate?
5. How do you turn an hourly GPU price into cost per million tokens?

Answers: 1 and 3 are in step 3 (6 × 4 = 24 in flight; the cache sets the slots); 2 is lesson 08; 4 is lesson 27; 5 is lesson 28.

## Step by step
### Step 1: the trace and the settings
The trace is `support_trace.jsonl` from lesson 07: 30 requests with prompt and answer lengths. It is **made up** by a seeded script for this course, so the numbers show the method, not a real company's traffic. The first lines show the format.

In [2]:
import json, math, random, heapq
from pathlib import Path
import numpy as np

lines = Path("../data/07/support_trace.jsonl").read_text().splitlines()
trace = [json.loads(line) for line in lines if line.strip()]
print("first lines of the file:", lines[:2])
mean_prompt = sum(r["prompt_tokens"] for r in trace) / len(trace)
mean_out = sum(r["output_tokens"] for r in trace) / len(trace)
print(f"{len(trace)} requests: mean prompt {mean_prompt:,.0f} tokens, mean answer {mean_out:,.0f} tokens")

first lines of the file: ['{"prompt_tokens": 1560, "output_tokens": 24}', '{"prompt_tokens": 1570, "output_tokens": 45}']
30 requests: mean prompt 1,553 tokens, mean answer 69 tokens


The next cell holds the settings. Three are **assumptions** for this exercise: the prefill time, the time per answer token at the target load, and the KV cache size. The KV cache size (150,000 tokens) is an assumed number, not computed. Lesson 27 replaces the assumptions with your own measurements. The other setting comes from the pinned vLLM image: the default `max_num_seqs` of 128 (lesson 12).

In [3]:
PREFILL_S = 0.25        # ASSUMPTION: prefill, the time to process the ~1,600-token prompt before the first token, on one replica
TPOT_S = 0.025          # ASSUMPTION: TPOT, time per output token (seconds per answer token) near the target load
SLOTS_CAP = 128         # max_num_seqs default of the pinned image (lesson 12)
KV_TOKENS = 150_000     # ASSUMPTION, not computed: KV cache tokens one replica can hold (lesson 05's formula)
TARGET_UTIL = 0.7       # headroom: run each replica at 70% of its slots at the peak
print(f"settings: prefill {PREFILL_S} s, {TPOT_S * 1000:.0f} ms per answer token, "
      f"cache {KV_TOKENS:,} tokens, target utilisation {TARGET_UTIL:.0%}")

settings: prefill 0.25 s, 25 ms per answer token, cache 150,000 tokens, target utilisation 70%


### Step 2: check the law with a queue simulation
The simulation is a first-in, first-out queue with a fixed number of slots. Requests arrive at random times (Poisson arrivals: random gaps around a steady average rate, from a fixed seed). Each one holds a slot for its prefill plus its answer tokens. The simulation counts two things over the run: the average number in the system, from the arrival and finish events, and the average time each request spent in the system, its sojourn (queue wait plus service). It then checks L against λ × W. The lab server of lesson 08 has 4 slots, so the test uses 4. Four slots is a toy size to verify the law; the fleet maths in step 3 uses 92.

In [4]:
def simulate(rate, slots, n=2000, seed=7):
    """FIFO queue with `slots` servers and Poisson arrivals at `rate` per second."""
    rng = random.Random(seed)
    free_at = [0.0] * slots                          # heap: when each slot is next free
    events, sojourn, t = [], [], 0.0
    for i in range(n):
        t += rng.expovariate(rate)                   # gap to the next arrival
        service = PREFILL_S + trace[i % len(trace)]["output_tokens"] * TPOT_S
        start = max(t, heapq.heappop(free_at))       # wait for the earliest free slot
        finish = start + service
        heapq.heappush(free_at, finish)
        events += [(t, 1), (finish, -1)]
        sojourn.append(finish - t)                   # arrival to finish: queue wait plus service
    events.sort()
    area, now, last = 0.0, 0, events[0][0]
    for when, step in events:                        # area under the number-in-system curve
        area, now, last = area + now * (when - last), now + step, when
    horizon = events[-1][0] - events[0][0]
    return {"L": area / horizon, "lam": n / horizon, "W": sum(sojourn) / n}

r = simulate(rate=1.2, slots=4)
print(f"L, average in system (measured):  {r['L']:.3f}")
print(f"lambda x W:                       {r['lam']:.3f} x {r['W']:.3f} = {r['lam'] * r['W']:.3f}")

L, average in system (measured):  2.682
lambda x W:                       1.219 x 2.201 = 2.682


**What you should see:** the two sides agree to the printed digits. The identity is exact here because the simulation counts every request from arrival to finish. On a real server it holds within a few percent, when the run is long and the queue is steady. The queue is real: 4 slots at about 2 seconds each can finish about 2 requests a second, so 1.2 requests a second keeps them about 60% busy and some requests still wait. Four slots is a toy size to verify the law; the fleet maths uses 92.

### Step 3: from QPS to concurrency to replicas
Each step is one line of arithmetic. The concurrency is L = λ × W. The slots a replica can really use are the smaller of `max_num_seqs` and what the cache holds: a request needs its prompt and answer tokens in the cache. The cache, not the flag, is usually the limit. Then the replicas are the in-flight requests divided by the usable slots, and one more replica covers a failure.

In [5]:
SECONDS_PER_REQUEST = PREFILL_S + mean_out * TPOT_S      # W, the time a request holds a slot
in_flight = PEAK_QPS * SECONDS_PER_REQUEST               # L = λ × W
tokens_per_request = mean_prompt + mean_out
slots_by_cache = int(KV_TOKENS // tokens_per_request)
slots = min(SLOTS_CAP, slots_by_cache)                   # the binding limit
usable = slots * TARGET_UTIL
replicas = math.ceil(in_flight / usable)
print(f"W = {SECONDS_PER_REQUEST:.2f} s per request; in flight at peak = {in_flight:,.0f}")
print(f"slots: cache allows {slots_by_cache}, flag allows {SLOTS_CAP} -> {slots} per replica")
print(f"usable at {TARGET_UTIL:.0%}: {usable:.0f}; replicas needed: {replicas}; with one spare: {replicas + 1}")
print(f"one slot per request at the measured W: {in_flight:,.0f} replicas; "
      f"batching cuts the fleet by {in_flight / (replicas + 1):.0f} times")

W = 1.98 s per request; in flight at peak = 19,800
slots: cache allows 92, flag allows 128 -> 92 per replica
usable at 70%: 64; replicas needed: 308; with one spare: 309
one slot per request at the measured W: 19,800 replicas; batching cuts the fleet by 64 times


**What you should see:** the cache, not `max_num_seqs`, sets the slots per replica: a replica holds 92 requests, not 128. A replica then runs at 70% of those slots, and the spare covers one failed replica. The 2.0 s guess in step 1 was rounded: the measured W is 1.98 s, so the one-slot-per-request count is 19,800, not 20,000. Lesson 13 shows what happens above the cache limit: requests wait or get preempted, which makes W longer, which makes L longer.

### Step 4: why 70% and not 95%
The formula gives the average. The queue waits depend on how close the replica runs to full. The Erlang C formula gives the chance that an arrival must wait, for a replica with `c` slots at utilisation ρ (rho: the share of slots busy). The mean wait and the 95th-percentile wait follow from it. The model is the standard **M/M/c** queue: c servers, Poisson arrivals and exponential (random) service times. Its formula assumes random service times; real answers vary in length, so the numbers are a guide, not a promise. Erlang C is a standard queueing result (Kleinrock, *Queueing Systems Vol. 1*, 1975, p. 103, as cited by the Wikipedia article "Erlang (unit)").

In [6]:
def erlang_c(c, rho):
    """Chance an arrival waits in an M/M/c queue: c servers, utilisation rho (0 to 1)."""
    if not 0 <= rho < 1:
        raise ValueError(f"utilisation {rho:.2f} must be below 1: the queue grows without bound")
    a = rho * c                                      # offered load in Erlangs: busy slots if nothing waited
    term, total = 1.0, 1.0                           # term = a^k / k!; total = sum for k < c
    for k in range(1, c):
        term *= a / k
        total += term
    top = term * a / c / (1 - rho)                   # (a^c / c!) × c / (c - a)
    return top / (total + top)

def mean_wait(c, rho, service_s):
    """Mean seconds an arrival waits for a slot: P(wait) × W / (c × (1 - rho))."""
    return erlang_c(c, rho) * service_s / (c * (1 - rho))

def p95_wait(c, rho, service_s):
    """95th-percentile seconds in the queue: the wait tail decays exponentially after the chance of waiting."""
    p = erlang_c(c, rho)
    return 0.0 if p <= 0.05 else math.log(p / 0.05) * service_s / (c * (1 - rho))

print(f"{'utilisation':>11} {'P(wait)':>8} {'mean wait (ms)':>15} {'p95 wait (ms)':>14} {'total (s)':>10}")
for rho in (0.5, 0.7, 0.8, 0.9, 0.95):
    w = mean_wait(slots, rho, SECONDS_PER_REQUEST)
    p95 = p95_wait(slots, rho, SECONDS_PER_REQUEST)
    print(f"{rho:>11.0%} {erlang_c(slots, rho):>8.2f} {w * 1000:>15.0f} {p95 * 1000:>14.0f} {SECONDS_PER_REQUEST + w:>10.2f}")

utilisation  P(wait)  mean wait (ms)  p95 wait (ms)  total (s)
        50%     0.00               0              0       1.98
        70%     0.00               0              0       1.98
        80%     0.02               3              0       1.98
        90%     0.23              50            332       2.03
        95%     0.52             225           1010       2.20


**What you should see:** the mean wait is 0 at 70%, 3 ms at 80%, 50 ms at 90% and 225 ms at 95%. The p95 wait is 0 at 80%, about 330 ms at 90% and about 1 s at 95%, so the tail is what a user feels first. That is why the target is 70% and not "as full as possible". The table ignores one effect: at high load each token gets slower (lesson 07), so the real knee, the utilisation above which waits climb steeply, comes a little earlier.

## At work
- **Cloudflare** routes each request by estimated in-flight prefill and decode tokens per endpoint, not by request count, because long prompts hold a slot far longer than short ones. → lesson 30 (routing) and lesson 29 (the gateway).
- **Google (GKE)** scales inference on queue size or batch size, not GPU utilisation, because a busy GPU can still have room for more requests. → lesson 31 (autoscaling).
- **Databricks** says to benchmark on your own deployment, since vendor numbers use other prompt and answer lengths. Your W is the input to this formula, so measure it at peak load. → lesson 27 (load tests that match reality).
- **Queue collapse:** when arrivals pass capacity, latency grows without bound, clients time out and retry, and the retries add load. Bound the queue and reject early. → lesson 33 (the reliability drill).

## Common mistakes
**Mistake 1: a fleet sized at the wrong utilisation.** Ask for the replicas at a rate the fleet cannot hold: the function refuses, and the message names the problem. This error is expected: 100 replicas cannot hold the 19,800 requests in flight, so the utilisation is above 1.

In [7]:
too_few = 100                                        # 100 replicas for about 19,800 requests in flight
rho = (in_flight / too_few) / slots
mean_wait(slots, rho, SECONDS_PER_REQUEST)

ValueError: utilisation 2.15 must be below 1: the queue grows without bound

The fix is the replica count from step 3, which keeps every replica below the knee. **Mistake 2: measuring W at one request.** A request alone on a replica is faster than one in a full batch, so a W measured with no load is too small and the fleet comes out too small. **Mistake 3: trusting the average.** L = λ × W says nothing about the p95; the tail needs the queue table and a load test (lesson 27).

## Production checklist
- Measure W at the target load, with your own prompt and answer lengths (lesson 27).
- Count slots from the cache budget, not from `max_num_seqs` alone.
- Run replicas below the utilisation knee, and keep one spare replica for a failure.
- Bound the queue and alert when it grows (lessons 31 and 33).
- Recompute when the mix of prompt and answer lengths changes.

## Try it
A prompt change makes answers 80 tokens long on average, so W and the in-flight count rise. Compute the new W, in-flight count and replica count with one spare, using the same settings.

In [8]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [9]:
new_out = 80
W2 = PREFILL_S + new_out * TPOT_S
in_flight2 = PEAK_QPS * W2
slots2 = min(SLOTS_CAP, int(KV_TOKENS // (mean_prompt + new_out)))
replicas2 = math.ceil(in_flight2 / (slots2 * TARGET_UTIL))
print(f"W {W2:.2f} s, in flight {in_flight2:,.0f}, slots {slots2}, replicas with spare {replicas2 + 1}")

W 2.25 s, in flight 22,500, slots 91, replicas with spare 355


## Recap
- **Little's law:** L = λ × W. Requests in flight equal the arrival rate times the time each one stays.
- **Concurrency to replicas:** divide the in-flight count by the slots a replica really has (the cache limit), at a utilisation below the knee; add one spare.
- **Measure W at peak load**, with your own lengths. A W measured alone is too small.
- **Queues grow fast near full:** the wait is small at 70% and climbs steeply at 95%.
- **Interview:** "How many replicas for 10,000 requests a second?" Strong answer: in flight = QPS × seconds per request; divide by the slots a replica can hold from its KV budget, at about 70% utilisation; add a spare; measure the seconds per request at peak load.